In [2]:
import requests

url = "https://api-web.nhle.com/v1/standings/now"

response = requests.get(url)

response.status_code # 200 indicates success

200

In [3]:
data = response.json()

In [4]:
data.keys()

dict_keys(['wildCardIndicator', 'standingsDateTimeUtc', 'standings'])

In [5]:
len(data['standings'])

32

In [6]:
standings = []

for team in data['standings']:
    standings.append({
        'team_abbrev': team['teamAbbrev']['default'],
        'season': str(team['seasonId']),
        'games_played': team['gamesPlayed'],
        'wins': team['wins'],
        'losses': team['losses'],
        'ot_losses': team['otLosses'],
        'points': team['points'],
        'goals_for': team['goalFor'],
        'goals_against': team['goalAgainst'],
        'home_wins': team['homeWins'],
        'away_wins': team['roadWins'],
        'streak_type': team['streakCode'],
        'streak_count': team['streakCount']
    })
standings

[{'team_abbrev': 'COL',
  'season': '20252026',
  'games_played': 82,
  'wins': 55,
  'losses': 16,
  'ot_losses': 11,
  'points': 121,
  'goals_for': 302,
  'goals_against': 203,
  'home_wins': 26,
  'away_wins': 29,
  'streak_type': 'W',
  'streak_count': 3},
 {'team_abbrev': 'CAR',
  'season': '20252026',
  'games_played': 82,
  'wins': 53,
  'losses': 22,
  'ot_losses': 7,
  'points': 113,
  'goals_for': 296,
  'goals_against': 240,
  'home_wins': 29,
  'away_wins': 24,
  'streak_type': 'W',
  'streak_count': 1},
 {'team_abbrev': 'DAL',
  'season': '20252026',
  'games_played': 82,
  'wins': 50,
  'losses': 20,
  'ot_losses': 12,
  'points': 112,
  'goals_for': 279,
  'goals_against': 226,
  'home_wins': 26,
  'away_wins': 24,
  'streak_type': 'W',
  'streak_count': 5},
 {'team_abbrev': 'BUF',
  'season': '20252026',
  'games_played': 82,
  'wins': 50,
  'losses': 23,
  'ot_losses': 9,
  'points': 109,
  'goals_for': 288,
  'goals_against': 241,
  'home_wins': 26,
  'away_wins': 24

In [7]:
!pip install pymysql


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
#connection string
import pymysql

#create a connection
conn = pymysql.connect(
    host = 'localhost',
    user = 'root',
    password = "sabi3601",
    database = 'nhl_analytics'
)

#create a cursor
cursor = conn.cursor() #this is how we make sure the connections
print("Connected to NHL Analytics database")

Connected to NHL Analytics database


In [9]:
create_standings_table = """
CREATE TABLE IF NOT EXISTS standings (
    standing_id INT AUTO_INCREMENT PRIMARY KEY,
    team_id INT NOT NULL,
    season VARCHAR(20) NOT NULL,
    games_played INT,
    wins INT,
    losses INT,
    ot_losses INT,
    points INT,
    goals_for INT,
    goals_against INT,
    home_wins INT,
    away_wins INT,
    streak_type VARCHAR(20),
    streak_count INT,
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
)
"""

cursor.execute(create_standings_table)

conn.commit()

print("Standings table created successfully")

Standings table created successfully


In [10]:
cursor.execute("SELECT team_id, team_abbrev FROM teams")

team_abbrev_to_id = {abbrev: team_id for team_id, abbrev in cursor.fetchall()}

team_abbrev_to_id

{'ANA': 18,
 'BOS': 8,
 'BUF': 4,
 'CAR': 2,
 'CBJ': 17,
 'CGY': 29,
 'CHI': 31,
 'COL': 1,
 'DAL': 3,
 'DET': 16,
 'EDM': 14,
 'FLA': 25,
 'LAK': 20,
 'MIN': 7,
 'MTL': 6,
 'NJD': 21,
 'NSH': 23,
 'NYI': 19,
 'NYR': 30,
 'OTT': 9,
 'PHI': 11,
 'PIT': 10,
 'SEA': 27,
 'SJS': 24,
 'STL': 22,
 'TBL': 5,
 'TOR': 28,
 'UTA': 15,
 'VAN': 32,
 'VGK': 13,
 'WPG': 26,
 'WSH': 12}

In [11]:
for row in standings:
    row['team_id'] = team_abbrev_to_id[row['team_abbrev']]

standings[0]

{'team_abbrev': 'COL',
 'season': '20252026',
 'games_played': 82,
 'wins': 55,
 'losses': 16,
 'ot_losses': 11,
 'points': 121,
 'goals_for': 302,
 'goals_against': 203,
 'home_wins': 26,
 'away_wins': 29,
 'streak_type': 'W',
 'streak_count': 3,
 'team_id': 1}

In [12]:
insert_query = """INSERT INTO standings
                    (team_id,season,games_played,wins,losses,ot_losses,points,goals_for,goals_against,home_wins,away_wins,streak_type,streak_count)
                    VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)"""

for i in standings:
   values = (i['team_id'],
         i['season'],
         i['games_played'],
         i['wins'],
         i['losses'],
         i['ot_losses'],
         i['points'],
         i['goals_for'],
         i['goals_against'],
         i['home_wins'],
         i['away_wins'],
         i['streak_type'],
         i['streak_count'])
   cursor.execute(insert_query, values)

conn.commit()

print(f"{len(standings)} standings rows inserted successfully")

32 standings rows inserted successfully


In [13]:
cursor.execute("SELECT COUNT(*) FROM standings")
cursor.fetchone()

(32,)